# SmartFood · Configuración y comprobación del entorno

Ejecuta este notebook antes de los notebooks 01–05. Comprueba el Python del kernel, revisa las versiones declaradas en `requirements.txt`, instala las dependencias que falten y verifica sus imports en procesos separados.

**Requisito previo:** abrir Jupyter con Python 3.10 o posterior. La instalación usa el mismo Python del kernel mediante `sys.executable`. Se recomienda seleccionar **Python (SmartFood)**, asociado a `.venv/`.

Este notebook no descarga datos ni entrena modelos. Distingue paquetes instalados de paquetes utilizables: una biblioteca puede estar instalada y fallar al cargar una DLL. Guarda el diagnóstico aunque algún import falle.

Consulta la [guía de notebooks](../docs/notebooks.md) y el [README](../README.md#ejecutar-el-proyecto).


## 1. Identificar el kernel y la raíz del proyecto


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
import json
import os
import platform
import subprocess
import sys

def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("smartfood")
if PROJECT_ROOT is None:
    raise RuntimeError("Inicia Jupyter desde SmartFood o desde su carpeta notebooks.")
if sys.version_info < (3, 10):
    raise RuntimeError("El proyecto requiere Python 3.10 o posterior. Cambia el kernel.")
os.chdir(PROJECT_ROOT)
REQUIREMENTS = PROJECT_ROOT / "requirements.txt"
if not REQUIREMENTS.is_file():
    raise FileNotFoundError(REQUIREMENTS)

IN_VIRTUAL_ENV = sys.prefix != sys.base_prefix
print("Proyecto:", PROJECT_ROOT)
print("Python del kernel:", sys.executable)
print("Versión:", platform.python_version())
print("Entorno virtual:", IN_VIRTUAL_ENV)
if not IN_VIRTUAL_ENV:
    print("La instalación se realizará en este Python. Se recomienda usar el kernel de .venv.")

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Configuracion comun: registrar avisos para el resumen de alertas.
from smartfood.entorno_notebook import configurar_notebook
configurar_notebook()


Proyecto: D:\portafolio\Analisis\Recommenders\SmartFood
Python del kernel: c:\Users\mauro\AppData\Local\Programs\Python\Python314\python.exe
Versión: 3.14.4
Entorno virtual: False
La instalación se realizará en este Python. Se recomienda usar el kernel de .venv.


## 2. Comprobar paquetes y versiones

`requirements.txt` contiene dependencias directas y la instalación editable del proyecto. Los rangos expresan requisitos mínimos; el diagnóstico guardará las versiones exactas del entorno.

El comprobador usa `packaging`. Si no está disponible, se instala primero como parte de los requisitos de este notebook.


In [2]:
try:
    from packaging.requirements import Requirement
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
         "--retries", "1", "--timeout", "30", "packaging>=23"],
        cwd=PROJECT_ROOT, check=True,
    )
    from packaging.requirements import Requirement

requirements = [
    Requirement(line.strip())
    for line in REQUIREMENTS.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith(("#", "-e"))
]

def check_versions():
    rows = []
    for requirement in requirements:
        if requirement.marker and not requirement.marker.evaluate():
            continue
        try:
            installed = metadata.version(requirement.name)
            compatible = requirement.specifier.contains(installed, prereleases=True)
            state = "OK" if compatible else "VERSIÓN INCOMPATIBLE"
        except metadata.PackageNotFoundError:
            installed, compatible, state = None, False, "FALTA INSTALAR"
        rows.append({
            "package": requirement.name,
            "required": str(requirement.specifier),
            "installed": installed,
            "compatible": compatible,
            "state": state,
        })
    return rows

def local_project_installed():
    try:
        distribution = metadata.distribution("smartfood")
        direct_url = json.loads(distribution.read_text("direct_url.json") or "{}")
        expected_url = PROJECT_ROOT.resolve().as_uri().rstrip("/")
        return (
            direct_url.get("dir_info", {}).get("editable", False)
            and direct_url.get("url", "").rstrip("/").casefold() == expected_url.casefold()
        )
    except (metadata.PackageNotFoundError, json.JSONDecodeError):
        return False

versions_before = check_versions()
for row in versions_before:
    print(f"{row['package']:18} {row['installed'] or 'ausente':12} {row['state']}")
print("SmartFood instalado desde esta carpeta:", local_project_installed())


numpy              2.3.5        OK
pandas             2.3.3        OK
scipy              1.18.1       OK
scikit-learn       1.9.1        OK
pyarrow            25.0.1       OK
matplotlib         3.11.2       OK
seaborn            0.13.2       OK
tqdm               4.70.1       OK
tabulate           0.10.0       OK
ipython            9.8.0        OK
ipywidgets         8.1.9        OK
jupyterlab         4.6.4        OK
ipykernel          7.1.0        OK
nbclient           0.11.0       OK
nbformat           5.11.1       OK
packaging          25.0         OK
SmartFood instalado desde esta carpeta: False


## 3. Instalar lo que falta

Si los rangos ya se cumplen y SmartFood está instalado desde esta carpeta, se omite la instalación. En caso contrario, se ejecuta `pip install -r requirements.txt` con el Python del kernel.

La instalación requiere acceso a internet cuando los paquetes no están en caché. Un fallo de descarga se mostrará como error y deberá resolverse antes de seguir. El proceso no actualiza automáticamente paquetes que ya cumplen los rangos.


In [3]:
installation_performed = (
    any(not row["compatible"] for row in versions_before)
    or not local_project_installed()
)
if installation_performed:
    command = [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--retries", "1", "--timeout", "30",
        "-r", str(REQUIREMENTS),
    ]
    print("Instalando en:", sys.executable, flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print("Instalación terminada. Reinicia el kernel y vuelve a ejecutar este notebook.")
else:
    print("Las dependencias y el paquete local ya cumplen los requisitos.")

versions_after = check_versions()
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    cwd=PROJECT_ROOT, capture_output=True, text=True, timeout=120,
)
print(pip_check.stdout.strip() or pip_check.stderr.strip())


Instalando en: c:\Users\mauro\AppData\Local\Programs\Python\Python314\python.exe
Instalación terminada. Reinicia el kernel y vuelve a ejecutar este notebook.
No broken requirements found.


## 4. Verificar que las bibliotecas se pueden cargar

Se comprueban imports reales en procesos separados para evitar que una instalación previa en el kernel o un fallo individual oculte el estado del resto. Cada import dispone de hasta 60 segundos.

La comprobación de scikit-learn incluye los módulos de métricas, NMF y SVM usados por el proyecto. No modifica políticas de Windows ni intenta resolver bloqueos de Control de aplicaciones. Un timeout indica que no se pudo verificar ese import dentro del plazo.


In [4]:
IMPORT_PROBES = {
    "numpy": "import numpy",
    "pandas": "import pandas",
    "scipy": "import scipy; import scipy.sparse",
    "scikit-learn": (
        "import sklearn; import sklearn.metrics; "
        "from sklearn.decomposition import NMF; from sklearn.svm import LinearSVC"
    ),
    "pyarrow": "import pyarrow; import pyarrow.parquet",
    "matplotlib": "import matplotlib; import matplotlib.pyplot",
    "seaborn": "import seaborn",
    "tqdm": "import tqdm",
    "tabulate": "import tabulate",
    "ipython": "import IPython; import IPython.display",
    "ipywidgets": "import ipywidgets",
    "jupyterlab": "import jupyterlab",
    "ipykernel": "import ipykernel",
    "nbclient": "import nbclient",
    "nbformat": "import nbformat",
    "packaging": "import packaging",
    "smartfood": "import smartfood; from smartfood.limpieza import limpiar_columnas",
}

import_results = []
for package, statement in IMPORT_PROBES.items():
    try:
        result = subprocess.run(
            [sys.executable, "-c", statement],
            cwd=PROJECT_ROOT, capture_output=True, text=True,
            encoding="utf-8", errors="replace", timeout=60,
            env={**os.environ, "PYTHONIOENCODING": "utf-8"},
        )
        state = "OK" if result.returncode == 0 else "ERROR"
        detail = (result.stderr or result.stdout).strip()
        row = {
            "package": package, "state": state,
            "returncode": result.returncode, "detail": detail,
        }
    except subprocess.TimeoutExpired:
        row = {
            "package": package, "state": "TIMEOUT",
            "returncode": None, "detail": "El import superó los 60 segundos.",
        }
    import_results.append(row)
    print(f"{package:18} {row['state']}", flush=True)
    if row["state"] != "OK":
        print(row["detail"][-1200:], flush=True)


numpy              OK
pandas             OK
scipy              OK
scikit-learn       OK
pyarrow            OK
matplotlib         OK
seaborn            OK
tqdm               OK
tabulate           OK
ipython            OK
ipywidgets         OK
jupyterlab         OK
ipykernel          OK
nbclient           OK
nbformat           OK
packaging          OK
smartfood          OK


## 5. Guardar el diagnóstico y decidir el siguiente paso

Se registran el Python activo, las versiones, la comprobación de dependencias y los imports. La disponibilidad de datos se muestra aparte y no altera el diagnóstico de bibliotecas.

**Entorno listo** significa que las comprobaciones de este notebook pasaron; no garantiza el desempeño de los modelos. Si se instalaron paquetes, reinicia el kernel y repite la comprobación antes de continuar.


In [5]:
all_imports_ok = all(row["state"] == "OK" for row in import_results)
all_versions_ok = all(row["compatible"] for row in versions_after)
environment_ready = all_imports_ok and all_versions_ok and pip_check.returncode == 0
timestamp = datetime.now(timezone.utc)
data_files = {
    name: (PROJECT_ROOT / "data" / "raw" / name).is_file()
    for name in ("recipes.csv", "reviews.csv", "recipes.parquet", "reviews.parquet")
}
report = {
    "checked_at_utc": timestamp.isoformat(),
    "project_root": str(PROJECT_ROOT),
    "python_executable": sys.executable,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "virtual_environment": IN_VIRTUAL_ENV,
    "installation_performed": installation_performed,
    "kernel_restart_recommended": installation_performed,
    "environment_ready": environment_ready,
    "versions": versions_after,
    "imports": import_results,
    "pip_check": {
        "returncode": pip_check.returncode,
        "stdout": pip_check.stdout, "stderr": pip_check.stderr,
    },
    "input_files_available": data_files,
}
report_dir = PROJECT_ROOT / "reports" / "entorno"
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "environment.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
frozen = subprocess.run(
    [sys.executable, "-m", "pip", "freeze", "--exclude-editable"],
    cwd=PROJECT_ROOT, capture_output=True, text=True, check=True, timeout=120,
)
report_path.with_suffix(".requirements.txt").write_text(
    "# Versiones del entorno comprobado. Instalar SmartFood por separado.\n" + frozen.stdout,
    encoding="utf-8",
)
print("Diagnóstico:", report_path)
print("Entorno listo:", environment_ready)
print("Datos disponibles:", data_files)
if installation_performed:
    print("Siguiente paso: reinicia el kernel y vuelve a ejecutar el notebook 00.")
elif not environment_ready:
    print("Siguiente paso: resolver los imports o versiones que no pasaron y repetir 00.")
elif not all(data_files[name] for name in ("recipes.parquet", "reviews.parquet")):
    print("Siguiente paso: preparar los datos Parquet mediante scripts/prepare_data.py.")
else:
    print("Siguiente paso: ejecutar 01_preparacion_datos.ipynb.")


Diagnóstico: D:\portafolio\Analisis\Recommenders\SmartFood\reports\entorno\environment.json
Entorno listo: True
Datos disponibles: {'recipes.csv': False, 'reviews.csv': False, 'recipes.parquet': True, 'reviews.parquet': True}
Siguiente paso: reinicia el kernel y vuelve a ejecutar el notebook 00.


## Alertas de la ejecución

La tabla agrupa los avisos de Python y conserva su detalle original. Si una celda falla antes de llegar aquí, puedes ejecutar esta celda de resumen para consultar las alertas ya registradas. Los errores y los mensajes de diagnóstico impresos no se incluyen automáticamente.


In [6]:
from smartfood.entorno_notebook import mostrar_alertas
mostrar_alertas(PROJECT_ROOT / "reports" / "alertas" / "00_configuracion_entorno.json")
